In [19]:
import chromadb
from sentence_transformers import SentenceTransformer


In [20]:
model = SentenceTransformer("all-MiniLM-L6-v2")

client = chromadb.PersistentClient(path="data/vector_store")
collection = client.get_collection(name="b2b_insights")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [21]:
def filtered_vector_search(query_text, source_filter=None, category_filter=None, top_k=5):
    
    query_embedding = model.encode(query_text).tolist()
    conditions = []
    
    if source_filter:
        conditions.append({"source": source_filter})
    if category_filter:
        conditions.append({"category": category_filter})
        
        
    where_filter = None
    if len(conditions) == 1:
        where_filter = conditions[0]  
    elif len(conditions) > 1:
        where_filter = {"$and": conditions} 
        

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k,
        where=where_filter,
        include=["documents", "metadatas", "distances", "embeddings"]
    )
    return results



In [22]:
if __name__ == "__main__":
    search_results = filtered_vector_search(
    query_text="sound", 
    top_k=50 # Increase this significantly!
)
    
    documents = search_results["documents"][0]
    metadatas = search_results["metadatas"][0]
    distances = search_results["distances"][0]

    print("\n===== SEARCH RESULTS =====\n")
    for i in range(len(documents)):
        print(f"Result {i+1}")
        print("-" * 50)
        print("Document:", documents[i])
        print("Metadata:", metadatas[i])
        print(f"Distance Score: {distances[i]:.4f}\n")


===== SEARCH RESULTS =====

Result 1
--------------------------------------------------
Document: My {product_purchased} is making strange noises and not functioning properly. I suspect there might be a hardware issue. Can you please help me with this? Thanks! -

http://www.sharespace.com/account I've followed online tutorials and community forums to troubleshoot the issue, but no luck so far. Cancellation request Billing inquiry
Metadata: {'title': 'Samsung Soundbar', 'Customer Age': 48, 'Customer Gender': 'Female', 'Ticket Channel': 'Phone', 'Date of Purchase': '2021-06-16'}
Distance Score: 1.3643

Result 2
--------------------------------------------------
Document: My {product_purchased} is making strange noises and not functioning properly. I suspect there might be a hardware issue. Can you please help me with this? https://pbs.twimg.com/media/CJGm I've performed a factory reset on my {product_purchased}, hoping it would resolve the problem, but it didn't help. Refund request Pro

In [ ]:
import umap
import hdbscan
import numpy as np
import pandas as pd

def cluster_retrieved_tickets(embeddings, documents, min_cluster_size=5):
    """
    Reduces vector dimensions and dynamically clusters text.
    """
    print(f"[INFO] Processing {len(embeddings)} retrieved tickets...")
    
    # 1. Convert the list of embeddings back to a Numpy array
    embeddings_array = np.array(embeddings)
    
    # 2. UMAP Dimensionality Reduction (384D -> 2D)
    safe_n_neighbors = min(15, len(embeddings_array) - 1)
    safe_n_neighbors = max(2, safe_n_neighbors) 

    print(f"[INFO] Reducing dimensions with UMAP (n_neighbors={safe_n_neighbors})...")
    reducer = umap.UMAP(
        n_neighbors=safe_n_neighbors, 
        n_components=2, 
        metric='cosine',
        random_state=42 
    )
    reduced_embeddings = reducer.fit_transform(embeddings_array)
    
    # 3. HDBSCAN Clustering
    print("[INFO] Finding patterns with HDBSCAN...")
    clusterer = hdbscan.HDBSCAN(
        min_cluster_size=min_cluster_size,
        metric='euclidean', 
        cluster_selection_method='eom' 
    )
    cluster_labels = clusterer.fit_predict(reduced_embeddings)
    
    # 4. Package the results cleanly into a Pandas DataFrame
    results_df = pd.DataFrame({
        'Document': documents,
        'Cluster_ID': cluster_labels,
        'X_coord': reduced_embeddings[:, 0],
        'Y_coord': reduced_embeddings[:, 1]
    })
    
    # Note: HDBSCAN labels noise/outliers as -1
    num_clusters = len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0)
    print(f"[INFO] Discovered {num_clusters} distinct topics/clusters.")
    
    return results_df



retrieved_embeddings = search_results["embeddings"][0] 
retrieved_documents = search_results["documents"][0]

clustered_data = cluster_retrieved_tickets(retrieved_embeddings, retrieved_documents)
print(clustered_data.head())

[INFO] Processing 50 retrieved tickets...
[INFO] Reducing dimensions with UMAP (n_neighbors=15)...


c:\Users\kaushal\Desktop\dsa\pythoin_ai_and_ml\major\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


[INFO] Finding patterns with HDBSCAN...
[INFO] Discovered 2 distinct topics/clusters.
                                            Document  Cluster_ID    X_coord  \
0  My {product_purchased} is making strange noise...           0   9.413851   
1  My {product_purchased} is making strange noise...           0   9.287440   
2  My {product_purchased} is making strange noise...           0   9.127297   
3  I'm having an issue with the {product_purchase...           1   7.042125   
4  My {product_purchased} is making strange noise...           0  10.651645   

    Y_coord  
0  8.460789  
1  8.068466  
2  8.635550  
3  7.256246  
4  9.815623  


In [24]:
import json
import os

def prepare_clusters_for_llm(clustered_df, save_path="data/llm_ready_clusters.json"):
    
    print("[INFO] Formatting data for LLM ingestion...")
    
    clean_df = clustered_df[clustered_df['Cluster_ID'] != -1]
    
    grouped_data = clean_df.groupby('Cluster_ID')['Document'].apply(list).to_dict()
    
    
    llm_payload = {}
    for cluster_id, documents in grouped_data.items():
        
        cluster_name = f"Cluster_{cluster_id}"
        llm_payload[cluster_name] = {
            "ticket_count": len(documents),
            "sample_tickets": documents
        }
        
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    with open(save_path, 'w', encoding='utf-8') as f:
        json.dump(llm_payload, f, indent=4)
        
    print(f"[INFO] Successfully saved {len(llm_payload)} clusters to {save_path}")
    return llm_payload


llm_ready_data = prepare_clusters_for_llm(clustered_data)

[INFO] Formatting data for LLM ingestion...
[INFO] Successfully saved 2 clusters to data/llm_ready_clusters.json
